<style>
.topic-header { background: linear-gradient(135deg, #e8f4f8 0%, #d4e8f0 100%); border-left: 4px solid #5ba4c9; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 10px 0; font-size: 15px; color: #1a3a4a; }
.concept-box { background: #eef6fa; border: 1px solid #c4dce8; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #2a4a5a; }
.try-it { background: #fef9e7; border: 1px solid #f0d87a; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #5a4a1a; }
.takeaway { background: #e8f5e8; border: 1px solid #a8d5a8; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #2a5a2a; }
.warning-box { background: #fdf0f0; border: 1px solid #e8b0b0; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #6a2a2a; }
.where-box { background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 0 8px 8px 0; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #4a3000; }
.fix-box { background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 0 8px 8px 0; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #1b5e20; }
.diagram-box { background: #f8f9fa; border: 2px solid #dee2e6; border-radius: 12px; padding: 24px; margin: 16px 0; text-align: center; }
.compare-table { width: 100%; border-collapse: collapse; margin: 12px 0; }
.compare-table th { background: #d4e8f0; color: #1a3a4a; padding: 10px 14px; text-align: left; border: 1px solid #c4dce8; }
.compare-table td { padding: 10px 14px; border: 1px solid #dee2e6; font-size: 13px; }
.compare-table tr:nth-child(even) { background: #f8fbfd; }
.section-divider { border: none; border-top: 2px solid #d4e8f0; margin: 25px 0; }
</style>

<div class='topic-header'>
<h1>E08 &middot; Day 3 &middot; RAG End-to-End &mdash; Ground the Model in YOUR Documents</h1>
<p><strong>GenAI for Engineering Managers &bull; Exercise 8 of 15 &bull; Opens Day 3</strong> &nbsp;|&nbsp; From an assistant that chats fluently to one that answers from your handbook and runbooks &mdash; with sources</p>
</div>

**Why this matters at your altitude.** Every "AI assistant for our docs" pitch your teams will bring you &mdash; support copilots, onboarding bots, runbook assistants &mdash; is, underneath, the pattern in this notebook: **Retrieval-Augmented Generation (RAG)**. In the next hour you will watch a raw model fail on questions about our own engineering handbook, then watch the *same model* answer them correctly &mdash; with citations &mdash; after we give it a retrieval layer. No fine-tuning, no training run, no data leaving the room. When a vendor quotes months for "training the AI on your documents", this session is your calibration for what that sentence should actually mean, cost, and take.

<div style="background: linear-gradient(135deg, #e8f4f8 0%, #c4dce8 100%); padding: 30px 32px; border-radius: 12px; margin-bottom: 20px;">
<h1 style="color: #1a3a4a; margin: 0; font-size: 28px;">A1 &middot; Day 4 &middot; From RAG + MCP to Agents</h1>
<p style="color: #3a6a8a; margin: 8px 0 0 0; font-size: 16px;">GenAI for Engineering Managers &mdash; Exercise 1 of 3 &middot; Facilitator-run (watch, or try alongside)</p>
<p style="color: #2a4a5a; margin: 14px 0 0 0; font-size: 14px; line-height: 1.6;">
<strong>The intent of this hour, stated plainly:</strong> everything you built in Day 3 is
<em>passive</em>. It retrieves and it answers. It never decides, and it never does.
This session finds the exact points where that breaks &mdash; three of them, each proven with
evidence you can see on screen &mdash; and then hands the model the ability to act.
By the end you will be able to say what an "agent" is without using the word "agent".
</p>
</div>

<div class='concept-box'>
<strong>How this hour is structured.</strong> We break the Day 3 assistant three times, on purpose.
Each break is <strong>proven, not asserted</strong> &mdash; we show the file, the hash, the date.
Then we fix them one at a time, and each fix reveals the next question. Nothing here is a slide.
</div>

<hr class='section-divider'>

## Part 1 &mdash; Rebuild the Day 3 Assistant

<div class='concept-box'>
Same corpus as E08 &mdash; handbook (.txt), runbook (.docx), returns policy (.pdf) &mdash; and the same
<strong>caged</strong> prompt: <em>answer ONLY from context, otherwise refuse</em>. That instruction was
correct in E08 and it is correct here. Remember it, because in Part 8 we take it away and the
consequences are larger than they look.
</div>

In [ ]:
%pip install -qU langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu pypdf docx2txt python-docx openai

In [ ]:
import os, json, hashlib, datetime, shutil

os.environ['OPENAI_API_KEY'] = 'PASTE_THE_KEY_SHARED_IN_SESSION_HERE'

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader, Docx2txtLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

MODEL  = 'gpt-4.1-nano'
llm    = ChatOpenAI(model=MODEL, temperature=0)
client = OpenAI()

print('Model ready:', MODEL)

In [ ]:
# Rebuild the E08 index and the incident queue — self-contained, run once.
%run ../data/setup_e08_docs.py
%run ../data/setup_incident_queue.py

docs = (TextLoader('../data/engineering_handbook.txt', encoding='utf-8').load()
        + Docx2txtLoader('../data/store_systems_runbook.docx').load()
        + PyPDFLoader('../data/supplier_returns_policy.pdf').load())

chunks    = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150).split_documents(docs)
retriever = FAISS.from_documents(chunks, OpenAIEmbeddings(model='text-embedding-3-small')) \
                 .as_retriever(search_kwargs={'k': 4})

CAGED_PROMPT = (
    "Answer using ONLY the context below. If the context does not contain the answer, "
    "reply exactly: 'The provided context does not contain information to answer this question.'"
    "\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
)

def ask_rag(question):
    """The Day 3 assistant, exactly as we left it."""
    context = "\n\n".join(c.page_content for c in retriever.invoke(question))
    return llm.invoke(CAGED_PROMPT.format(context=context, question=question)).content.strip()

print(f'{len(chunks)} chunks indexed. The Day 3 assistant is back.')
print()
print('Sanity check —')
print('Q: What is the standard return window for a marketplace item?')
print('A:', ask_rag('What is the standard return window for a marketplace item?'))

<hr class='section-divider'>

## Part 2 &mdash; Break 1: It Has No Memory

<div class='concept-box'>
The most ordinary thing a person does in a conversation: ask a follow-up. Watch what happens when
the follow-up contains the word <strong>"that"</strong>.
</div>

In [ ]:
print('TURN 1')
q1 = 'What is the standard return window for a marketplace item?'
print('Q:', q1)
print('A:', ask_rag(q1))

print('\nTURN 2 — an ordinary human follow-up')
q2 = 'Does that apply to electronics?'
print('Q:', q2)
print('A:', ask_rag(q2))

<div class='warning-box'>
<strong>Before you diagnose this, rule out the obvious explanation.</strong> Maybe the answer simply
is not in our documents. Let us check &mdash; same question, asked without the word "that".
</div>

In [ ]:
q2_standalone = 'What is the return window for electronics specifically?'
print('Q:', q2_standalone)
print('A:', ask_rag(q2_standalone))

<div class='takeaway'>
<strong>The fact was there the whole time.</strong> "Electronics carry an extended 45-day window" is
sitting in the PDF. The system did not fail to <em>know</em> it &mdash; it failed to know
<strong>what "that" referred to</strong>.<br><br>
Every call to <code>ask_rag()</code> starts from nothing. It embedded the word "that", searched for
chunks that resemble "that", and found nothing useful. <strong>This is not a retrieval bug or a model
weakness. It is the absence of a conversation.</strong>
</div>

### The cheap fix &mdash; carry the history yourself

<div class='concept-box'>
Nothing clever required: keep a list of what was said, and paste it into the next prompt.
</div>

In [ ]:
history = []   # our entire "memory", for now

def ask_with_history(question):
    transcript = "\n".join(f"{role}: {text}" for role, text in history)
    context    = "\n\n".join(c.page_content for c in retriever.invoke(question))
    prompt = (
        "You are an internal assistant. Use the conversation so far to understand what the "
        "user's question refers to, then answer using ONLY the context provided.\n\n"
        f"Conversation so far:\n{transcript or '(nothing yet)'}\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    )
    answer = llm.invoke(prompt).content.strip()
    history.append(("User", question))
    history.append(("Assistant", answer))
    return answer

print('TURN 1')
print('Q:', q1); print('A:', ask_with_history(q1))
print('\nTURN 2 — the same follow-up that just failed')
print('Q:', q2); print('A:', ask_with_history(q2))

<div class='takeaway'>
<strong>Fixed &mdash; and notice how little it took.</strong> No new model, no new index. We simply
stopped throwing the conversation away.
</div>

<div class='warning-box'>
<strong>But look at what we just signed up for.</strong> Every turn appends to <code>history</code>,
and the <em>whole</em> transcript rides along in every future prompt. Let us measure the growth
rather than hand-wave it.
</div>

In [ ]:
transcript_chars = sum(len(t) for _, t in history)
print(f'After 2 turns, the transcript we resend every time is {transcript_chars:,} characters.')
print(f'Projected at that rate — 20 turns: ~{transcript_chars * 10:,} chars   '
      f'|  100 turns: ~{transcript_chars * 50:,} chars')
print()
print('And the harder problem:')
print('  history lives in a Python variable.')
print('  Restart this kernel and the assistant forgets the customer entirely.')

<div class='topic-header'>
<strong>&#128279; Gap left open &rarr; Exercise 2.</strong> Memory that grows without limit and dies on
restart is not memory, it is a variable. Real memory has to be <em>bounded</em> and it has to
<em>survive a restart</em>. That is Exercise 2 &mdash; we will come back to this exact problem.
</div>

<hr class='section-divider'>

## Part 3 &mdash; Break 2: It Cannot Know What Day It Is

<div class='concept-box'>
A model is a snapshot of text frozen at training time. It has no clock. Our documents have no clock
either. So any question anchored to <em>now</em> has no possible source.
</div>

In [ ]:
for q in ["What is today's date?",
          "Shipment SHP-88121 was due on 11 August 2026. Is it overdue as of today?",
          "How many days until the peak-season change freeze begins?"]:
    print('Q:', q)
    print('A:', ask_rag(q))
    print('-' * 72)

print("\nMeanwhile, this machine knows perfectly well:")
print("  today is", datetime.date.today().isoformat())

<div class='takeaway'>
<strong>Read the third answer carefully.</strong> The first two refuse honestly. The third does
something worse &mdash; it reaches for the freeze date in the handbook and produces a confident
non-answer about "15 days into November", because it was asked for a <em>duration from now</em> and
has no "now" to subtract from.<br><br>
The date is available on the machine the notebook is running on. The model simply has no way to reach
out and ask for it. <strong>It is not missing knowledge. It is missing a hand.</strong>
</div>

<hr class='section-divider'>

## Part 4 &mdash; Break 3: It Cannot Do Anything

<div class='concept-box'>
The two failures so far were about knowing. This one is about <strong>doing</strong>, and it is the
one that matters most to a manager evaluating an AI proposal.<br><br>
We give the model an operations-assistant persona &mdash; exactly the framing a vendor demo uses &mdash;
and ask it to escalate a real ticket from our incident queue. <strong>We hash the file before and
after</strong>, so nobody has to take anyone's word for what happened.
</div>

In [ ]:
TICKETS = '../data/incident_tickets.json'
shutil.copy(TICKETS, '../data/_incident_tickets_backup.json')   # so we can prove the reset later

def ticket_fingerprint():
    raw = open(TICKETS, 'rb').read()
    return hashlib.sha256(raw).hexdigest()[:16]

def ticket_status(tid):
    data = json.load(open(TICKETS))
    return next(t.get('status') for t in data['tickets'] if t['id'] == tid)

print('Queue on disk:', [t['id'] for t in json.load(open(TICKETS))['tickets']])
print('INC-30021 status :', ticket_status('INC-30021'))
print('File fingerprint :', ticket_fingerprint())

In [ ]:
before_hash   = ticket_fingerprint()
before_status = ticket_status('INC-30021')

reply = llm.invoke([
    ('system', 'You are the Store Systems operations assistant for a large retailer. '
               'You help the on-call engineer manage the incident queue. Be concise and confident.'),
    ('human',  "Escalate ticket INC-30021 to SEV-1 and assign it to the Store Systems Reliability team. "
               "Then confirm."),
]).content.strip()

print('THE ASSISTANT SAYS:')
print(' ', reply)

print('\nTHE FILE SAYS:')
print(f'  fingerprint before : {before_hash}')
print(f'  fingerprint after  : {ticket_fingerprint()}')
print(f'  INC-30021 status   : {before_status}  ->  {ticket_status("INC-30021")}')
print(f'\n  FILE ACTUALLY CHANGED: {before_hash != ticket_fingerprint()}')

<div class='warning-box'>
<strong>Stop on this cell. It is the most important one in the session.</strong><br><br>
The assistant reported the escalation as complete. The fingerprint is unchanged. The status is
still exactly what it was. <strong>Nothing happened.</strong> No error was raised, no warning was
printed, and the sentence it produced is indistinguishable from the sentence a working system
would produce.<br><br>
If this were a vendor demo, everyone in the room would have seen a successful escalation.
</div>

<div class='takeaway'>
<strong>The manager's rule, and it is worth writing down:</strong> when an AI system claims it did
something, <strong>the sentence is not the evidence. The system of record is the evidence.</strong>
Ask to see the row change. Ask for the audit log. A fluent confirmation costs the model nothing.
</div>

<hr class='section-divider'>

## Part 5 &mdash; The Fix: Give It Tools

<div class='concept-box'>
All three failures have the same shape. The model can only produce <strong>text</strong>, and text
cannot read a clock, look up a live record, or write to a file.<br><br>
A <strong>tool</strong> is an ordinary Python function we describe to the model. The model cannot run
it &mdash; it can only <em>request</em> that we run it, and we decide whether to comply. That
distinction is the whole safety story, and Exercise 3 is built on it.
</div>

### Step 5.1 &mdash; One tool, and watch it choose

In [ ]:
def get_todays_date():
    """Return today's date from the machine's clock."""
    return {'today': datetime.date.today().isoformat()}

TOOL_SPECS = [{
    'type': 'function',
    'function': {
        'name': 'get_todays_date',
        'description': "Get today's date. Use for any question about what day it is, or anything "
                       "relative to 'today', 'now', or 'currently'.",
        'parameters': {'type': 'object', 'properties': {}},
    },
}]

TOOL_FUNCS = {'get_todays_date': get_todays_date}
print('1 tool registered:', list(TOOL_FUNCS))

In [ ]:
# ── AI VERBOSE MODE ──────────────────────────────────────────────────
# Flip this to watch the agent decide, instead of only seeing its answer.
VERBOSE = False

def set_verbose(on=True):
    """Switch the AI verbose trace on or off."""
    global VERBOSE
    VERBOSE = on
    print(f"AI verbose mode: {'ON  - every decision shown' if on else 'OFF - answers only'}")


def _rule(title):
    print(f"\n+{'-' * 74}")
    print(f"| {title}")
    print(f"+{'-' * 74}")


def run_agent(question, system='You are the Store Systems operations assistant.',
              specs=None, funcs=None, max_steps=6, show_trace=True):
    """Plan -> Act -> Observe, repeated until the model stops asking for tools."""
    specs = TOOL_SPECS if specs is None else specs
    funcs = TOOL_FUNCS if funcs is None else funcs
    messages = [{'role': 'system', 'content': system},
                {'role': 'user',   'content': question}]
    tokens_in = tokens_out = 0

    if VERBOSE:
        _rule(f'QUESTION: {question}')
        print(f'| system prompt : {system[:86]}')
        print(f'| tools offered : {[s["function"]["name"] for s in specs]}')

    for step in range(max_steps):
        response    = client.chat.completions.create(model=MODEL, messages=messages, tools=specs)
        msg         = response.choices[0].message
        tokens_in  += response.usage.prompt_tokens
        tokens_out += response.usage.completion_tokens

        if VERBOSE:
            _rule(f'TURN {step + 1}   .   model reads {len(messages)} messages   .   '
                  f'{response.usage.prompt_tokens} tokens in / {response.usage.completion_tokens} out')
            if msg.content:
                print(f'| model writes  : {msg.content.strip()[:300]}')

        if not msg.tool_calls:                      # nothing more to do — this is the answer
            if VERBOSE:
                print('| DECISION      : no tool needed - this is the final answer')
                print(f'\n   session total: {tokens_in} tokens in / {tokens_out} out')
            return msg.content.strip()

        if VERBOSE:
            print(f'| DECISION      : call {len(msg.tool_calls)} tool(s)')

        messages.append(msg)
        for call in msg.tool_calls:
            args   = json.loads(call.function.arguments or '{}')
            result = funcs[call.function.name](**args)
            if VERBOSE:
                print(f'|   ---> {call.function.name}({json.dumps(args)})')
                print(f'|   <--- {json.dumps(result, default=str)[:220]}')
            elif show_trace:
                print(f'   [step {step + 1}] model called {call.function.name}({args}) '
                      f'-> {json.dumps(result, default=str)[:110]}')
            messages.append({'role': 'tool', 'tool_call_id': call.id,
                             'content': json.dumps(result, default=str)})
    return '(gave up - too many steps)'


q = "What is today's date?"
print('Q:', q)
print('  -- trace --')
answer = run_agent(q)
print('A:', answer)
print('\nMachine clock says:', datetime.date.today().isoformat())

<div class='takeaway'>
<strong>Break 2 is closed</strong> &mdash; but look at the trace, not the answer. We never told the model
to call <code>get_todays_date</code>. We described the tool and asked a question. <strong>The model
decided the tool was relevant.</strong><br><br>
That decision is the entire difference between software that calls an LLM and an LLM that calls
software.
</div>

### Step 5.2 &mdash; Three tools, and now it has to pick the right one

<div class='concept-box'>
One tool proves nothing &mdash; with a single option, "choosing" is trivial. Add two more, including a
live lookup against the shipment feed from E10, and watch the selection actually matter.
</div>

In [ ]:
def get_shipment_status(shipment_id):
    """Live status for an inbound shipment — the system of record from E10."""
    for s in json.load(open('../data/shipments.json')):
        if s['shipment_id'].upper() == shipment_id.upper():
            return s
    return {'error': f'no shipment {shipment_id}'}

def get_ticket(ticket_id):
    """Read one incident ticket from the queue."""
    for t in json.load(open(TICKETS))['tickets']:
        if t['id'].upper() == ticket_id.upper():
            return {'id': t['id'], 'title': t['title'], 'store': t['store'],
                    'status': t.get('status'), 'severity': t.get('severity_reported')}
    return {'error': f'no ticket {ticket_id}'}

TOOL_SPECS += [
    {'type': 'function', 'function': {
        'name': 'get_shipment_status',
        'description': 'Look up the CURRENT live status and ETA of an inbound shipment by its id.',
        'parameters': {'type': 'object',
                       'properties': {'shipment_id': {'type': 'string'}},
                       'required': ['shipment_id']}}},
    {'type': 'function', 'function': {
        'name': 'get_ticket',
        'description': 'Read the current state of one incident ticket by its id.',
        'parameters': {'type': 'object',
                       'properties': {'ticket_id': {'type': 'string'}},
                       'required': ['ticket_id']}}},
]
TOOL_FUNCS.update({'get_shipment_status': get_shipment_status, 'get_ticket': get_ticket})
print('Toolbox now:', list(TOOL_FUNCS))

In [ ]:
for q in ["What is today's date?",
          'Is shipment SHP-88121 delayed?',
          'What is the current status of ticket INC-30021?']:
    print('Q:', q)
    print('  -- trace --')
    print('A:', run_agent(q))
    print('=' * 72)

<div class='takeaway'>
<strong>Three questions, three different tools, zero instructions from us about which to use.</strong>
The model read the descriptions we wrote and matched them to intent.<br><br>
<strong>The management consequence:</strong> those tool <em>descriptions</em> are now production
configuration. A vague description causes wrong-tool selection, and that is a bug you fix in English,
not in Python. Ask your teams who reviews them.
</div>

<div class='concept-box'>
And notice the shipment answer: <strong>SHP-88121, delayed, ETA 18 August, weather hold at DC 6094.</strong>
That is the exact question Day 3 could not answer &mdash; the stale memo said 11 August and RAG had no
way to know better. <strong>One tool call closed the gap E08 left open.</strong>
</div>

<hr class='section-divider'>

## Part 6 &mdash; The Tool That Changes Something

<div class='concept-box'>
Every tool so far only <em>reads</em>. Reading is safe. Now we add one that <strong>writes</strong>,
and re-run the exact request that silently did nothing in Part 4.
</div>

In [ ]:
def update_ticket(ticket_id, status, note=''):
    """Write a new status to an incident ticket. This one CHANGES things."""
    data = json.load(open(TICKETS))
    for t in data['tickets']:
        if t['id'].upper() == ticket_id.upper():
            t['status'] = status
            t.setdefault('audit', []).append(
                {'status': status, 'note': note, 'at': datetime.datetime.now().isoformat(timespec='seconds')})
            json.dump(data, open(TICKETS, 'w'), indent=2)
            return {'ok': True, 'ticket': ticket_id, 'new_status': status}
    return {'ok': False, 'error': f'no ticket {ticket_id}'}

TOOL_SPECS.append({'type': 'function', 'function': {
    'name': 'update_ticket',
    'description': 'Change the status of an incident ticket. This modifies the system of record.',
    'parameters': {'type': 'object',
                   'properties': {'ticket_id': {'type': 'string'},
                                  'status':    {'type': 'string'},
                                  'note':      {'type': 'string'}},
                   'required': ['ticket_id', 'status']}}})
TOOL_FUNCS['update_ticket'] = update_ticket
print('Toolbox now:', list(TOOL_FUNCS))

In [ ]:
before_hash   = ticket_fingerprint()
before_status = ticket_status('INC-30021')

print('Q: Escalate ticket INC-30021 to status "escalated", note "POS sync backlog climbing". Then confirm.')
print('  -- trace --')
answer = run_agent('Escalate ticket INC-30021 to status "escalated", '
                   'note "POS sync backlog climbing". Then confirm.')
print('A:', answer)

print('\nTHE FILE SAYS:')
print(f'  fingerprint before : {before_hash}')
print(f'  fingerprint after  : {ticket_fingerprint()}')
print(f'  INC-30021 status   : {before_status}  ->  {ticket_status("INC-30021")}')
print(f'\n  FILE ACTUALLY CHANGED: {before_hash != ticket_fingerprint()}')

audit = next(t for t in json.load(open(TICKETS))['tickets'] if t['id'] == 'INC-30021').get('audit')
print('  audit trail written:', json.dumps(audit))

<div class='takeaway'>
<strong>Put this next to Part 4 and read both answers out loud.</strong> The sentences are nearly
identical &mdash; "the ticket has been escalated". In Part 4 the fingerprint did not move.
Here it did, the status changed, and an audit entry exists with a timestamp.<br><br>
<strong>Same words. Opposite reality.</strong> The only thing that changed is that the model was given
a hand, and someone wrote down what it did.
</div>

<div class='warning-box'>
<strong>And now the uncomfortable part.</strong> Nobody approved that escalation. The model decided,
called the tool, and the write landed &mdash; in one cell, with no gate. Hold that thought until Part 9.
</div>

<hr class='section-divider'>

## Part 7 &mdash; The Flip: RAG as a Cage vs RAG as a Tool

<div class='concept-box'>
In E08 we <em>caged</em> the model: <strong>answer ONLY from context, otherwise refuse.</strong> That was
the right call &mdash; we wanted refusals instead of invention, and we got them.<br><br>
But that instruction has a cost we never measured. Ask something that needs
<strong>one ingredient from our documents and one from the model's own knowledge</strong>.
</div>

In [ ]:
combo_questions = [
    'Our marketplace return window is in the policy. Is that more or less generous than the typical '
    'e-commerce industry standard, and why might that matter competitively?',
    'Explain our POS_SYNC_LAG remediation steps to a new engineer who has never heard of a consumer '
    'group, using an everyday analogy.',
]

print('THE CAGE — answer ONLY from context')
print('=' * 72)
for q in combo_questions:
    print('\nQ:', q[:95] + '...')
    print('A:', ask_rag(q))

<div class='warning-box'>
<strong>Both refused.</strong> And not because the facts are missing &mdash; the 30-day window, the 45-day
electronics extension, and the restart-one-at-a-time rule are all in the corpus. They refused because
the answer also needed something the model already knew, and
<strong>we declared everything it knew inadmissible.</strong>
</div>

<div class='concept-box'>
Now the flip. <strong>Same index. Same chunks. Same embeddings.</strong> The only change: instead of
force-feeding retrieved text into a locked prompt, we offer retrieval as a <em>tool the model may
choose to call</em>.
</div>

In [ ]:
def policy_lookup(query):
    """Search the internal handbook, runbook and returns policy. The E08 index, as a tool."""
    hits = retriever.invoke(query)
    return {'passages': [c.page_content for c in hits]}

TOOL_SPECS.append({'type': 'function', 'function': {
    'name': 'policy_lookup',
    'description': 'Search internal company documents: the engineering handbook, the store-systems '
                   'runbook, and the marketplace returns policy. Use for any company-specific fact.',
    'parameters': {'type': 'object',
                   'properties': {'query': {'type': 'string'}},
                   'required': ['query']}}})
TOOL_FUNCS['policy_lookup'] = policy_lookup

print('THE TOOL — same index, model free to combine it with what it knows')
print('=' * 72)
for q in combo_questions:
    print('\nQ:', q[:95] + '...')
    print('  -- trace --')
    print('A:', run_agent(
        q, system='You are an internal engineering assistant. Use policy_lookup for company-specific '
                  'facts, and combine what it returns with your own general knowledge when that helps '
                  'the reader.'))
    print('=' * 72)

<div class='takeaway'>
<strong>This is the transition, in one comparison.</strong> We did not change the model, the documents,
the chunking or the embeddings. We changed <em>the model's relationship to the documents</em>.<br><br>
<strong>Cage:</strong> the documents are the only permitted world. Reliable, auditable, and unable to
explain, compare, teach or contextualise. It can only quote.<br>
<strong>Tool:</strong> the documents are one source the model consults <em>on purpose</em>, then reasons
over. It can teach a new engineer using an analogy that appears nowhere in the runbook.<br><br>
Watch the traces: in both questions the model called <code>policy_lookup</code>
<strong>more than once</strong> &mdash; it decided a single lookup was not enough. Nobody programmed that.
</div>

<div class='warning-box'>
<strong>This is a trade, not an upgrade.</strong> Uncaging buys reasoning and pays for it in
auditability &mdash; part of the answer now comes from the model's own knowledge, which no citation
covers. For a regulated disclosure, keep the cage. For an internal assistant that must explain things,
open it. <strong>The decision is per use case, and it is a management decision, not a technical one.</strong>
</div>

<hr class='section-divider'>

## Part 8 &mdash; What We Actually Built: the Loop

<div class='concept-box'>
Everything since Part 5 has run through one function, <code>run_agent</code>, and it is about
fifteen lines. Read them again with fresh eyes:<br><br>
<strong>1. PLAN</strong> &mdash; send the question and the tool list; the model decides what it needs.<br>
<strong>2. ACT</strong> &mdash; if it requested a tool, <em>we</em> run it. The model never executes anything.<br>
<strong>3. OBSERVE</strong> &mdash; the result goes back into the conversation.<br>
<strong>4. REPEAT</strong> &mdash; until the model stops asking and answers.<br><br>
That is the entire trick. Everything your teams will call "agentic AI" is this loop plus
engineering around it.
</div>

In [ ]:
print('A question that needs more than one tool, in sequence:')
print()
q = ('Check whether shipment SHP-88121 has arrived yet given what day it is today, '
     'and tell me what our runbook says about stale inventory feeds.')
print('Q:', q)
print('  -- trace --')
print('A:', run_agent(q))

<div class='try-it'>
<strong>Read the trace, not the answer.</strong> Count the steps. The model was not given a plan &mdash;
it discovered that it needed the date, then the shipment, then the runbook, and it sequenced them
itself. Between steps it <em>observed</em> and re-decided.<br><br>
That is the difference between a workflow you wrote and an agent that plans.
</div>

<hr class='section-divider'>

## Part 9 &mdash; AI Verbose: Watching It Decide

<div class='concept-box'>
Everything so far showed you a one-line trace and an answer. Now we open the whole thing up.
<code>set_verbose(True)</code> prints, for every single turn: <strong>what the model was given, what it
wrote, what it decided, which tool it called with which arguments, exactly what came back, and what it
cost in tokens.</strong>
</div>

<div class='warning-box'>
<strong>Say this to the room before you run it, because the distinction matters.</strong><br><br>
This is <em>not</em> the model's inner monologue. Models of this class do not expose their hidden
reasoning, and anything claiming to show you "the AI's thoughts" is usually showing you generated text
<em>about</em> thinking, not the thinking itself.<br><br>
What you are about to see is better, because it is real: <strong>the decision trace.</strong> What the
model received, what it chose to do, what the world sent back, and what it did with that. Those are
the things you can audit, log, replay and hold a vendor to. Inner monologue is not.
</div>

In [ ]:
set_verbose(True)

In [ ]:
answer = run_agent('Given what day it is today, has shipment SHP-88121 arrived at store 4479 yet?')
print('\nFINAL ANSWER:', answer)

<div class='try-it'>
<strong>Four things to point at on screen:</strong>
<ol style='margin:6px 0 0 18px;'>
<li><strong>"model reads N messages"</strong> — it grows every turn. The conversation is resent in full,
every time. That number <em>is</em> your bill.</li>
<li><strong>DECISION lines</strong> — the model chose the date tool before the shipment tool, because it
needed today before it could judge "arrived yet". Nobody ordered that sequence.</li>
<li><strong>The <code>&lt;---</code> lines</strong> — the raw data that came back, before the model
touched it. This is the ground truth for the next point.</li>
<li><strong>Tokens in / out per turn</strong> — the cost of autonomy, itemised.</li>
</ol>
</div>

### The reason this switch earns its place

<div class='concept-box'>
Compare the raw tool output against the final sentence. The model had the real ETA in hand &mdash; did
the answer keep it? Let us not eyeball it.
</div>

In [ ]:
truth = get_shipment_status('SHP-88121')
print('What the tool returned :', truth['current_eta'], f"({truth['status']})")
print('What the answer said   :', answer[:200])
print()
kept = truth['current_eta'] in answer or '18 August' in answer or 'August 18' in answer
print('Final answer preserved the real ETA:', kept)
if not kept:
    print()
    print('  ^ The model had the correct date and drifted it in the last sentence.')
    print('    Compact mode would have shown you only that fluent final answer.')

<div class='takeaway'>
<strong>That check is the argument for verbose mode.</strong> The failure, when it happens, is not in
retrieval and not in the tool &mdash; both did their job perfectly, and you can see them doing it. It is
in <strong>the last mile</strong>, where the model turns correct data into a sentence.<br><br>
In compact mode you would have seen only a confident, well-formatted answer with a wrong date in it.
<strong>Verbose mode puts the tool's raw output and the model's claim side by side, where a human can
compare them.</strong> That is the difference between a system you can audit and one you can only
believe.<br><br>
<em>(This drift is intermittent &mdash; the model does not do it every run. Re-run the cell a few times.
"Intermittent" is precisely what makes it dangerous: it will pass your demo and fail in week three.)</em>
</div>

### Switching it off

<div class='concept-box'>
Verbose is for teaching, debugging and incident review. In a demo to stakeholders it is noise.
One call turns it off, and everything downstream reverts to answers only.
</div>

In [ ]:
set_verbose(False)

print()
print('Same question, verbose off:')
print(run_agent('Given what day it is today, has shipment SHP-88121 arrived at store 4479 yet?',
                show_trace=False))

<div class='takeaway'>
<strong>Same code, same tools, same model &mdash; two very different things to look at.</strong><br><br>
The second output is what your users will see. The first is what your on-call engineer needs at 3 a.m.
when someone asks why the agent did what it did. <strong>Build both from day one</strong>, and keep the
trace in your logs even when the switch is off in the UI &mdash; because you cannot reconstruct a
decision after the fact from an answer alone.
</div>

<hr class='section-divider'>

## Part 10 &mdash; The Question That Should Worry You: How Much Autonomy?

<div class='concept-box'>
Return to Part 6. The model escalated a real ticket, and <strong>nobody approved it</strong>. It was
right that time. The question a manager has to answer is not "can it act?" &mdash; you have now seen
that it can &mdash; but <strong>"how far do we let it act before a human sees it?"</strong>
</div>

<table class='compare-table'>
<tr><th>Setting</th><th>What the model may do</th><th>Fits</th></tr>
<tr><td><strong>0 &mdash; Suggest</strong></td><td>Text only. A human does everything.</td><td>Regulated advice, anything customer-visible</td></tr>
<tr><td><strong>1 &mdash; Read</strong></td><td>Call read-only tools. Cannot change state.</td><td>Diagnostics, triage assistance, search</td></tr>
<tr><td><strong>2 &mdash; Propose</strong></td><td>Read freely; draft an action and <em>stop</em> for approval.</td><td><strong>Most production systems today</strong></td></tr>
<tr><td><strong>3 &mdash; Act, reversible</strong></td><td>Take actions that can be undone, and log them.</td><td>Tagging, routing, internal ticket updates</td></tr>
<tr><td><strong>4 &mdash; Act, irreversible</strong></td><td>Send, pay, delete, notify customers.</td><td>Rare. Needs a hard business case</td></tr>
</table>

<div class='warning-box'>
<strong>What we built today sits at 3</strong> &mdash; and we arrived there by accident, simply by adding
a tool. That is exactly how it happens in real teams. Nobody decides to ship an autonomous system;
somebody adds a write tool to a read-only assistant on a Thursday.<br><br>
<strong>The design-review question:</strong> <em>"Which dial setting is this, and who signed off on it?"</em>
</div>

<hr class='section-divider'>

## Recap

<table class='compare-table'>
<tr><th>Part</th><th>What we proved</th><th>Manager takeaway</th></tr>
<tr><td>2</td><td>Follow-up failed; the same fact answered when asked standalone</td><td>A knowledge gap and a memory gap look identical from the outside</td></tr>
<tr><td>3</td><td>No answer about today; a confident non-answer on a date delta</td><td>A model has no clock — anything about "now" needs a tool</td></tr>
<tr><td>4</td><td>"Escalation complete" — fingerprint unchanged, status unchanged</td><td><strong>The sentence is not the evidence. The system of record is</strong></td></tr>
<tr><td>5</td><td>The model chose the right tool from three, unprompted</td><td>Tool descriptions are production config, written in English</td></tr>
<tr><td>6</td><td>Same request, same words — fingerprint changed, audit written</td><td>Ask to see the row change, not the confirmation message</td></tr>
<tr><td>7</td><td>Cage refused both; tool answered both, looking up twice</td><td>Uncaging buys reasoning and costs auditability — a per-use-case call</td></tr>
<tr><td>8</td><td>Multi-step question sequenced without a plan from us</td><td>Plan → Act → Observe. That is all an agent is</td></tr>
<tr><td>9</td><td>Verbose mode: every message, decision, tool result and token</td><td>You cannot audit a decision from the answer alone — log the trace</td></tr>
<tr><td>10</td><td>We reached dial setting 3 by adding one function</td><td>Autonomy creeps in through tools, not through decisions</td></tr>
</table>

<div class='concept-box'>
<strong>Glossary</strong>
<table class='compare-table'>
<tr><td><strong>Tool</strong></td><td>A normal function, described to the model in English, that the model may <em>request</em></td></tr>
<tr><td><strong>Tool call</strong></td><td>The model's request. It never executes anything itself</td></tr>
<tr><td><strong>Tool selection</strong></td><td>The model matching intent to a description — the first agentic behaviour</td></tr>
<tr><td><strong>Agent loop</strong></td><td>Plan &rarr; Act &rarr; Observe, repeated until done (also called ReAct)</td></tr>
<tr><td><strong>RAG-as-cage</strong></td><td>Retrieval forced into a locked prompt; the model may use nothing else</td></tr>
<tr><td><strong>RAG-as-tool</strong></td><td>Retrieval offered as one option the model may choose and combine</td></tr>
<tr><td><strong>Decision trace</strong></td><td>The auditable record of what a model received, chose and was told — not its hidden reasoning</td></tr>
<tr><td><strong>Autonomy dial</strong></td><td>How far the system may act before a human sees it (0–4)</td></tr>
</table>
</div>

In [ ]:
# Housekeeping — put the incident queue back exactly as we found it,
# so this notebook can be re-run from a clean state.
shutil.copy('../data/_incident_tickets_backup.json', TICKETS)
os.remove('../data/_incident_tickets_backup.json')
print('Incident queue restored. INC-30021 status:', ticket_status('INC-30021'))
print('Fingerprint:', ticket_fingerprint())

<div class='topic-header'>
<strong>&#128279; The two gaps we leave</strong><br><br>
<strong>Gap 1 &mdash; memory (Exercise 2).</strong> We fixed amnesia with a Python list that grows on
every turn and vanishes when the kernel restarts. That is not memory. Exercise 2 makes it
<em>bounded</em> and makes it <em>survive a restart</em> &mdash; and adds the other direction, letting the
agent look <strong>outward</strong> at the live web rather than only inward at our documents.<br><br>
<strong>Gap 2 &mdash; the gate (Exercise 3).</strong> In Part 6 the agent changed our system of record and
nobody approved it. Exercise 3 puts a human in front of that write &mdash; a real Approve / Reject
button on a real interface &mdash; and proves that a rejected action leaves the file untouched.
</div>